# AST Transfer Learning

This notebook implements the transfer learning pipeline using the Audio Spectrogram Transformer (AST) architecture on the GUITAR-FX dataset.

In [ ]:
import os
import sys
import time
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# Ensure project root is in sys.path
sys.path.append('..')
from src.data.dataset_ast import FxDatasetAST
from src.data.datasplit import DataSplit
from src.models.ast_models import ASTModel
from src.utils.device import get_device

%matplotlib inline

## 1. Configuration
Select the dataset scenario (mono/poly - disc/cont) and define transfer learning hyperparameters.

In [15]:
# --- SCENARIO SELECTOR ---
# 'mono_disc' | 'mono_cont' | 'poly_disc' | 'poly_cont'
DATASET_TYPE = 'poly_cont'

# Global hyperparameters
SEED = 42
EPOCHS_PROBE = 200     # Epochs for training only the linear classifier head
BATCH_SIZE = 64       # AST consumes more vRAM, use a smaller batch size
LR_PROBE = 1e-3
DEVICE_NAME = None
NUM_WORKERS = 0
CACHE_IN_RAM = False
TARGET_LENGTH = 1024  # AST expected input frames (adjust based on audio length)

# Scenario-specific dataset root and parameters
if DATASET_TYPE == 'mono_disc':
    DATASET_ROOT = '../datasets/GUITAR-FX/mel_16/Mono_Discrete'
elif DATASET_TYPE == 'mono_cont':
    DATASET_ROOT = '../datasets/GUITAR-FX/mel_16/Mono_Continuous'
elif DATASET_TYPE == 'poly_disc':
    DATASET_ROOT = '../datasets/GUITAR-FX/mel_16/Poly_Discrete'
elif DATASET_TYPE == 'poly_cont':
    DATASET_ROOT = '../datasets/GUITAR-FX/mel_16/Poly_Continuous'
else:
    raise ValueError(f"Unknown DATASET_TYPE: {DATASET_TYPE}")

EXCL_FOLDERS = ['_NoFX_mono', '_NoFX_mono_preprocessed', '_NoFX_poly', '_NoFX_poly_preprocessed', 'MT2']

# Output directories
RESULTS_DIR = "../results"
SCENARIO_RESULTS_DIR = os.path.join(RESULTS_DIR, f"ast_transfer", DATASET_TYPE)

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(SCENARIO_RESULTS_DIR, exist_ok=True)

print(f"[{DATASET_TYPE.upper()}] Configuration loaded.")

[POLY_CONT] Configuration loaded.


## 2. Data Loading
Uses the `FxDatasetAST` class optimized for the AST log-mel standard format.

In [16]:
device = get_device(DEVICE_NAME)
print(f"Selected compute device: {device}")

print("Loading dataset...")
dataset = FxDatasetAST(
    root=DATASET_ROOT,
    excl_folders=EXCL_FOLDERS,
    cache_in_ram=CACHE_IN_RAM,
    target_length=TARGET_LENGTH
)
dataset.init_dataset()

print(f"Total dataset samples: {len(dataset):,}")
print(f"Number of effect classes: {dataset.num_fx}")

import os
from src.data.transfer import group_splits, source_group
from torch.utils.data.sampler import SubsetRandomSampler
from torch.utils.data import DataLoader
from src.data.datasplit import custom_collate_fn

print("Computing robust splits based on source audio...")
groups = []
for _, label, path in dataset.audiosamples_labels:
    filename = os.path.basename(path).replace(".npy", ".wav")
    effect = dataset.label_to_fx[label]
    groups.append(source_group(filename, effect))

assignments = group_splits(groups, seed=SEED)

train_indices = [i for i, g in enumerate(groups) if assignments[g] == "train"]
val_indices = [i for i, g in enumerate(groups) if assignments[g] == "validation"]
test_indices = [i for i, g in enumerate(groups) if assignments[g] == "test"]

print(f"Split sizes: Train {len(train_indices)} | Val {len(val_indices)} | Test {len(test_indices)}")

train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, sampler=SubsetRandomSampler(train_indices), collate_fn=custom_collate_fn, num_workers=NUM_WORKERS)
val_loader = DataLoader(dataset, batch_size=BATCH_SIZE, sampler=SubsetRandomSampler(val_indices), collate_fn=custom_collate_fn, num_workers=NUM_WORKERS)
test_loader = DataLoader(dataset, batch_size=BATCH_SIZE, sampler=SubsetRandomSampler(test_indices), collate_fn=custom_collate_fn, num_workers=NUM_WORKERS)

Selected compute device: cuda
Loading dataset...
Total dataset samples: 130,000
Number of effect classes: 13
Computing robust splits based on source audio...
Split sizes: Train 93402 | Val 10064 | Test 26534


## 3. Model Setup
Loads the pre-trained AST model and sets up the loss criterion.

In [17]:
import gc
import sys
from tqdm.auto import tqdm

class MultiprocessingErrorFilter:
    def __init__(self, original_stderr):
        self.original_stderr = original_stderr
    def write(self, message):
        if "AssertionError: can only test a child process" not in message and "Exception ignored in" not in message:
            self.original_stderr.write(message)
    def flush(self):
        self.original_stderr.flush()

sys.stderr = MultiprocessingErrorFilter(sys.stderr)

print("Loading pre-trained AST model...")
model = ASTModel(
    label_dim=dataset.num_fx,
    input_tdim=TARGET_LENGTH,
    imagenet_pretrain=True,
    audioset_pretrain=True
).to(device)

# Bypass the final classification head to output the 768-d embeddings directly
model.mlp_head = nn.Identity()
model.eval()

# Freeze all parameters
for param in model.parameters():
    param.requires_grad = False

def extract_embeddings(loader, desc):
    embeds_all, labels_all = [], []
    with torch.no_grad():
        for data in tqdm(loader, desc=desc):
            if data is None: continue
            inputs, labels, _, _, _ = data

            with torch.amp.autocast('cuda'):
                embeds = model(inputs.to(device))

            embeds_all.append(embeds.cpu())
            labels_all.append(labels)

    return torch.cat(embeds_all), torch.cat(labels_all)

print("Extracting embeddings (this runs only once)...")
train_embs, train_lbls = extract_embeddings(train_loader, "Train")
val_embs, val_lbls = extract_embeddings(val_loader, "Validation")
test_embs, test_lbls = extract_embeddings(test_loader, "Test")

sys.stderr = sys.__stderr__

# Free up GPU memory for the linear probing phase
del model
gc.collect()
torch.cuda.empty_cache()
print("Extraction complete. GPU memory cleared (and warnings suppressed successfully!).")

Loading pre-trained AST model...
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=1212
Extracting embeddings (this runs only once)...


Train:   0%|          | 0/1460 [00:00<?, ?it/s]

Validation:   0%|          | 0/158 [00:00<?, ?it/s]

Test:   0%|          | 0/415 [00:00<?, ?it/s]

Extraction complete. GPU memory cleared (and warnings suppressed successfully!).


## 4. Linear Probing (Transfer Learning)
Freeze the transformer backbone and train only the MLP classifier head for the target domain.

In [20]:
import numpy as np
from torch.utils.data import TensorDataset
import copy

EPOCHS_PROBE = EPOCHS_PROBE
PATIENCE = 15
SEEDS = [42, 101, 202, 303, 404]
test_accuracies = []

# Create fast TensorDatasets holding embeddings in memory
train_dataset = TensorDataset(train_embs, train_lbls)
val_dataset = TensorDataset(val_embs, val_lbls)
test_dataset = TensorDataset(test_embs, test_lbls)

print(f"Starting Linear Probing (Max Epochs: {EPOCHS_PROBE}, Patience: {PATIENCE})...")

for seed in SEEDS:
    torch.manual_seed(seed)

    # Linear head: 768 (AST embedding dim) -> num_fx classes
    head = nn.Linear(768, dataset.num_fx).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=LR_PROBE, weight_decay=0.01)

    # Scheduler: Diminui a taxa de aprendizado se a acurácia parar de subir
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    criterion = nn.CrossEntropyLoss()

    train_dl = DataLoader(train_dataset, batch_size=1024, shuffle=True)
    val_dl = DataLoader(val_dataset, batch_size=1024, shuffle=False)
    test_dl = DataLoader(test_dataset, batch_size=1024, shuffle=False)

    best_val_acc = 0.0
    best_weights = None
    epochs_no_improve = 0

    # Training Loop
    for epoch in range(EPOCHS_PROBE):
        head.train()
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = head(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

        # Validation Phase
        head.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_dl:
                x, y = x.to(device), y.to(device)
                logits = head(x)
                correct += (logits.argmax(1) == y).sum().item()
                total += y.size(0)

        val_acc = correct / max(total, 1)
        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = copy.deepcopy(head.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        # Early Stopping
        if epochs_no_improve >= PATIENCE:
            break

    # Evaluation on Test Set using best validation weights
    head.load_state_dict(best_weights)
    head.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_dl:
            x, y = x.to(device), y.to(device)
            logits = head(x)
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)

    test_acc = (correct / max(total, 1)) * 100
    test_accuracies.append(test_acc)
    # Mostra em qual época ele atingiu o melhor resultado (parou antes das 200)
    best_epoch = epoch + 1 - epochs_no_improve
    print(f"Seed {seed} -> Test Accuracy: {test_acc:.2f}% (Best Val Epoch: {best_epoch})")

# Final Results Output
test_accuracies = np.array(test_accuracies)
print("\n" + "=" * 45)
print("AST Linear Probing Results (5 Seeds)")
print(f"Mean Accuracy: {test_accuracies.mean():.2f}% +/- {test_accuracies.std():.2f}%")
print("=" * 45)


Starting Linear Probing (Max Epochs: 200, Patience: 15)...
Seed 42 -> Test Accuracy: 89.81% (Best Val Epoch: 79)
Seed 101 -> Test Accuracy: 90.28% (Best Val Epoch: 83)
Seed 202 -> Test Accuracy: 90.61% (Best Val Epoch: 103)
Seed 303 -> Test Accuracy: 90.28% (Best Val Epoch: 139)
Seed 404 -> Test Accuracy: 90.33% (Best Val Epoch: 123)

AST Linear Probing Results (5 Seeds)
Mean Accuracy: 90.26% +/- 0.26%


## 5. Metrics & Artifacts Generation
Evaluate the best model from the last seed and export standard metrics (JSON, confusion matrix) matching the repository.

In [21]:
import time
import json
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from src.utils.metrics import plot_confusion_matrix

# We use the best_weights from the last seed to generate the detailed matrix
head.load_state_dict(best_weights)
head.eval()

true_lbls, predicted = [], []
with torch.no_grad():
    for x, y in test_dl:
        x, y = x.to(device), y.to(device)
        logits = head(x)
        predicted.extend(logits.argmax(1).cpu().tolist())
        true_lbls.extend(y.cpu().tolist())

# Generate sklearn metrics safely using label_to_fx aligned by index
labels_idx = list(range(dataset.num_fx))
fx_names = [dataset.label_to_fx[i] for i in labels_idx]

report = classification_report(true_lbls, predicted, labels=labels_idx, target_names=fx_names, output_dict=True, zero_division=0)
metrics = {
    "accuracy": float(accuracy_score(true_lbls, predicted)),
    "macro_precision": report["macro avg"]["precision"],
    "macro_recall": report["macro avg"]["recall"],
    "macro_f1": report["macro avg"]["f1-score"],
    "classification_report": report,
    "confusion_matrix": confusion_matrix(true_lbls, predicted, labels=labels_idx).tolist(),
    "multi_seed_mean_accuracy": float(test_accuracies.mean()),
    "multi_seed_std_accuracy": float(test_accuracies.std()),
    "training_config": {
        "max_epochs": EPOCHS_PROBE,
        "patience": 15,
        "learning_rate": LR_PROBE,
        "best_epoch_seed_404": best_epoch
    }
}

# Create output directory similar to the colleague's script
run_folder = Path(SCENARIO_RESULTS_DIR) / time.strftime("probe_%Y%m%d_%H%M%S")
run_folder.mkdir(parents=True, exist_ok=True)

# Save JSON
(run_folder / "test_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"Metrics saved to {run_folder}/test_metrics.json")

# Plot and save Confusion Matrix
cm_path = str(run_folder / "confusion_matrix.png")
plot_confusion_matrix(np.asarray(metrics["confusion_matrix"]), fx_names, cm_path, normalize=True)
print(f"Confusion Matrix saved to {cm_path}")

# Save the Linear Probe model weights to the disk
model_path = str(run_folder / f"best_probe{DATASET_TYPE}.pth")
torch.save(head.state_dict(), model_path)
print(f"Model weights saved to {model_path}")

print("\nQuick Overview (Last Seed):")
print(f"Accuracy : {metrics['accuracy']*100:.2f}%")
print(f"Macro F1 : {metrics['macro_f1']*100:.2f}%")
print(f"Macro Pre: {metrics['macro_precision']*100:.2f}%")
print(f"Macro Rec: {metrics['macro_recall']*100:.2f}%")

Metrics saved to ../results/ast_transfer/poly_cont/probe_20261002_165042/test_metrics.json
Confusion Matrix saved to ../results/ast_transfer/poly_cont/probe_20261002_165042/confusion_matrix.png
Model weights saved to ../results/ast_transfer/poly_cont/probe_20261002_165042/best_probepoly_cont.pth

Quick Overview (Last Seed):
Accuracy : 90.33%
Macro F1 : 90.31%
Macro Pre: 90.31%
Macro Rec: 90.34%
